# 05 - Demo agente LangGraph

Demo local del grafo Hunter + SDR usando filas reales de `analysis/top50.csv`.

In [ ]:
from pathlib import Path
import os
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent.resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Cargar .env local sin imprimir secretos.
env_path = ROOT / '.env'
if env_path.exists():
    for line in env_path.read_text(encoding='utf-8').splitlines():
        if not line or line.strip().startswith('#') or '=' not in line:
            continue
        key, value = line.split('=', 1)
        os.environ.setdefault(key.strip(), value.strip())

import pandas as pd
from IPython.display import Markdown, display

from src.agents.graph import compiled_graph
from src.db.models import DEFAULT_DB_PATH

# Reset determinístico de DB local ignorada para evitar duplicados entre ejecuciones del notebook.
for suffix in ['', '-wal', '-shm']:
    db_path = Path(str(DEFAULT_DB_PATH) + suffix)
    if db_path.exists():
        db_path.unlink()

top50 = pd.read_csv(ROOT / 'analysis' / 'top50.csv')
assert os.environ.get('GROQ_API_KEY'), 'Falta GROQ_API_KEY en entorno o .env para clasificar replies reales con Groq'
print('Setup OK: graph importado, DB local reseteada, GROQ_API_KEY cargada sin mostrar valor')

## Grafo Mermaid

In [ ]:
mermaid = '''
flowchart LR
    A[Top50 real] --> B{Tier}
    B -->|A| C[Brief Hunter Sr / Slack]
    B -->|B| D[Chequeo duplicado]
    D --> E[Email D0]
    E --> F[Clasificar reply]
    F --> G{Router}
    G -->|agendar| H[WhatsApp gate + Slack]
    G -->|nurture| I[Slack nurture]
    G -->|descartar| J[Slack descarte / WhatsApp bloqueado]
'''
print(mermaid)
display(Markdown('```mermaid\n' + mermaid.strip() + '\n```'))


## Corridas con filas reales

In [ ]:
def brand_row(brand_id, **overrides):
    row = top50.loc[top50['brand_id'].eq(brand_id)]
    assert not row.empty, f'No existe {brand_id} en top50.csv'
    data = row.iloc[0].dropna().to_dict()
    data.update(overrides)
    return data

cases = [
    {
        'name': 'Tier A - Hunter Sr',
        'brand_id': 'Brand_0002',
        'state': {
            'brand_data': brand_row('Brand_0002'),
            'tier': 'A',
            'ya_contactado': False,
            'reply_recibido': None,
            'clasificacion': None,
            'decision': '',
            'log_razonamiento': [],
        },
    },
    {
        'name': 'Tier B - Agendar',
        'brand_id': 'Brand_0145',
        'state': {
            'brand_data': brand_row('Brand_0145', contacto_email='demo@example.com'),
            'tier': 'B',
            'ya_contactado': False,
            'reply_recibido': 'Sí me interesa, podemos agendar una llamada esta semana con mi equipo de ecommerce.',
            'clasificacion': None,
            'decision': '',
            'log_razonamiento': [],
        },
    },
    {
        'name': 'Tier B - Opt-out / descarte',
        'brand_id': 'Brand_0826',
        'state': {
            'brand_data': brand_row('Brand_0826', contacto_email='demo@example.com'),
            'tier': 'B',
            'ya_contactado': False,
            'reply_recibido': 'Por favor no me vuelvan a escribir, no me interesa.',
            'clasificacion': None,
            'decision': '',
            'log_razonamiento': [],
        },
    },
]

results = []
for case in cases:
    result = compiled_graph.invoke(case['state'])
    results.append({'case': case['name'], 'brand_id': case['brand_id'], 'result': result})
    print(f"\n=== {case['name']} | {case['brand_id']} ===")
    print(f"decision: {result.get('decision')}")
    print(f"clasificacion: {result.get('clasificacion')}")
    print('log_razonamiento:')
    for line in result.get('log_razonamiento', []):
        print(f"- {line}")

## Resumen estructurado

In [ ]:
summary = pd.DataFrame([
    {
        'case': item['case'],
        'brand_id': item['brand_id'],
        'tier': item['result'].get('tier'),
        'decision': item['result'].get('decision'),
        'log_steps': len(item['result'].get('log_razonamiento', [])),
    }
    for item in results
])
summary